# SaaS Churn & Retention Analysis
## Milestone 2: Foundation (Definitions, MRR, & Topline Churn)

**Author:** Anuj Saini  
**Date:** July 7, 2026  

This notebook calculates and visualizes the company's Current Monthly Recurring Revenue (MRR) and monthly churn rate trends over the last 12 months.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from dotenv import load_dotenv
import os

# Set design style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Load environment variables from .env file
load_dotenv(dotenv_path="../.env")

### 1. Database Connection and Schema Inspection

We establish a connection to the analytical database and pull the raw subscription dataset.

In [ ]:
# Retrieve credentials
host = os.getenv("host")
port = os.getenv("port")
database = os.getenv("database")
username = os.getenv("username")
password = os.getenv("password")

# Establish connection
conn = psycopg2.connect(
    host=host,
    port=port,
    database=database,
    user=username,
    password=password
)

# Query all subscriptions
query = "SELECT * FROM saas.legacy_subscriptions;"
df = pd.read_sql(query, conn)

# Convert date columns
df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])
df['cancelled_at'] = pd.to_datetime(df['cancelled_at'])

print(f"Dataset successfully loaded. Total rows: {df.shape[0]}")
df.head()

### 2. Current MRR Breakdown by Plan Tier

We compute the company's Current Monthly Recurring Revenue (MRR) using the sum of the `mrr` column for paid subscriptions that are currently active (`status = 'active'`).

In [ ]:
# Filter for active paid subscriptions
active_df = df[(df['status'] == 'active') & (df['plan'] != 'free') & (df['mrr'] > 0)]

# Group by plan tier
mrr_breakdown = active_df.groupby('plan').agg(
    active_subscriptions=('id', 'count'),
    total_mrr=('mrr', 'sum'),
    average_mrr=('mrr', 'mean')
).reset_index()

mrr_breakdown['total_mrr'] = mrr_breakdown['total_mrr'].round(2)
mrr_breakdown['average_mrr'] = mrr_breakdown['average_mrr'].round(2)
mrr_breakdown['pct_total_mrr'] = (mrr_breakdown['total_mrr'] / mrr_breakdown['total_mrr'].sum() * 100).round(2)

mrr_breakdown

In [ ]:
# Plot MRR by Plan Tier
plt.figure(figsize=(8, 5))
ax = sns.barplot(data=mrr_breakdown, x='plan', y='total_mrr', hue='plan', palette='viridis', legend=False)
plt.title('Current Monthly Recurring Revenue (MRR) by Plan Tier', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Plan Tier', fontsize=12)
plt.ylabel('Total MRR ($)', fontsize=12)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')

# Add labels on top of bars
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 1000,
            f'${height:,.2f}',
            ha="center", va="bottom", fontsize=10, fontweight='semibold')

plt.tight_layout()
plt.show()

### 3. Monthly Churn Rate Trend (Last 12 Months)

We calculate the monthly paid subscription and MRR churn rates. The formula is:
- **Subscription Churn Rate** = (Subscriptions churned in month M) / (Active subscriptions at the start of month M)
- **Gross MRR Churn Rate** = (MRR churned in month M) / (MRR at the start of month M)

In [ ]:
# Month boundaries range for the last 12 months
months = pd.date_range(start='2025-05-01', end='2026-04-30', freq='ME')
monthly_metrics = []

for m in months:
    m_end = m
    m_start = m - pd.offsets.MonthBegin(1)
    prev_end = m_start - pd.Timedelta(days=1)
    
    # Starting active paid subscriptions (active at end of previous month)
    starting_subs = df[
        (df['plan'] != 'free') & 
        (df['status'] != 'trial') & 
        (df['start_date'] <= prev_end) & 
        ((df['end_date'].isnull()) | (df['end_date'] > prev_end))
    ]
    start_subs_count = len(starting_subs)
    start_mrr_sum = starting_subs['mrr'].sum()
    
    # Churned paid subscriptions during current month
    churned_subs = df[
        (df['plan'] != 'free') & 
        (df['status'] == 'churned') & 
        (df['end_date'] >= m_start) & 
        (df['end_date'] <= m_end)
    ]
    churned_subs_count = len(churned_subs)
    churned_mrr_sum = churned_subs['mrr'].sum()
    
    # Calculate rates
    sub_churn_rate = (churned_subs_count / start_subs_count * 100) if start_subs_count > 0 else 0
    mrr_churn_rate = (churned_mrr_sum / start_mrr_sum * 100) if start_mrr_sum > 0 else 0
    
    # Free plan subscriber metrics
    starting_free_subs = df[
        (df['plan'] == 'free') & 
        (df['start_date'] <= prev_end) & 
        ((df['end_date'].isnull()) | (df['end_date'] > prev_end))
    ]
    churned_free = df[
        (df['plan'] == 'free') & 
        (df['status'] == 'churned') & 
        (df['end_date'] >= m_start) & 
        (df['end_date'] <= m_end)
    ]
    free_churn_rate = (len(churned_free) / len(starting_free_subs) * 100) if len(starting_free_subs) > 0 else 0
    
    monthly_metrics.append({
        'Month': m.strftime('%Y-%m'),
        'StartingPaidSubs': start_subs_count,
        'StartingPaidMRR': round(start_mrr_sum, 2),
        'ChurnedPaidSubs': churned_subs_count,
        'ChurnedPaidMRR': round(churned_mrr_sum, 2),
        'SubChurnRatePct': round(sub_churn_rate, 2),
        'MRRChurnRatePct': round(mrr_churn_rate, 2),
        'StartingFreeSubs': len(starting_free_subs),
        'FreeChurnRatePct': round(free_churn_rate, 2)
    })
    
monthly_df = pd.DataFrame(monthly_metrics)
monthly_df

In [ ]:
# Plot paid subscription and MRR churn rate trend
plt.figure(figsize=(12, 6))
plt.plot(monthly_df['Month'], monthly_df['SubChurnRatePct'], marker='o', label='Subscription Churn Rate (%)', color='#1f77b4', linewidth=2.5)
plt.plot(monthly_df['Month'], monthly_df['MRRChurnRatePct'], marker='s', label='Gross MRR Churn Rate (%)', color='#ff7f0e', linewidth=2.5, linestyle='--')
plt.title('Monthly Paid Churn Rate Trend (Last 12 Months)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Churn Rate (%)', fontsize=12)
plt.gca().yaxis.set_major_formatter('{x:.1f}%')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 4. Paid Churn vs. Free Churn Comparison

Free users are not committed financially, showing more erratic retention profiles.

In [ ]:
# Compare paid churn rate vs free churn rate
plt.figure(figsize=(12, 6))
plt.plot(monthly_df['Month'], monthly_df['SubChurnRatePct'], marker='o', label='Paid Plan Churn (%)', color='#1f77b4', linewidth=2)
plt.plot(monthly_df['Month'], monthly_df['FreeChurnRatePct'], marker='x', label='Free Plan Churn (%)', color='#2ca02c', linewidth=2, linestyle=':')
plt.title('Paid Plan Churn vs Free Plan Churn Rate', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Churn Rate (%)', fontsize=12)
plt.gca().yaxis.set_major_formatter('{x:.1f}%')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()